# 🦴 Ясны Хугарлын Оношилгооны Pipeline v2

**Нэгтгэсэн хамгийн сайн загваруудын дамжлага:**

| Алхам | Загвар | Зорилго | Үр дүн |
|---|---|---|---|
| 1 | `mura_classifier.pt` | Хугарал илрүүлэлт | fracture / normal |
| 2 | `hybrid_unet_final.pth` | Сегментчлэл (ResNet50 U-Net + TTA) | binary mask, Dice=0.604 |
| 3 | Morphological heuristic | Severity ангилал | Grade 0–3 |
| 4 | `grazped_healing.pt` | Эдгэрэлтийн таамаглал | долоо хоног + 95% CI |

### Drive дээр байх ёстой файлууд
```
MyDrive/
  mura_classifier.pt
  hybrid_unet_final.pth
  grazped_healing.pt
```

> Runtime → Change runtime type → **T4 GPU**

In [ ]:
# ── Cell 1: GPU шалгах ───────────────────────────────────────────────────────
import torch
print('GPU:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
else:
    print('⚠ GPU байхгүй — Runtime > Change runtime type > T4 GPU')

In [ ]:
# ── Cell 2: Сангуудыг суулгах ────────────────────────────────────────────────
!pip install -q segmentation-models-pytorch albumentations scikit-image
print('✓ Суулгалт дууслаа')

In [ ]:
# ── Cell 3: Google Drive холбох ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('✓ Drive холбогдлоо')

In [ ]:
# ── Cell 4: Import ───────────────────────────────────────────────────────────
import os, json, math, warnings
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from scipy import ndimage
from skimage import measure

warnings.filterwarnings('ignore')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('✓ Бүх сан ачаалагдлаа')
print('Device:', DEVICE)

In [ ]:
# ── Cell 5: Тохиргоо ─────────────────────────────────────────────────────────
DRIVE_ROOT = '/content/drive/MyDrive'

MURA_CKPT   = f'{DRIVE_ROOT}/mura_classifier.pt'
HYBRID_CKPT = f'{DRIVE_ROOT}/hybrid_unet_final.pth'
HEAL_CKPT   = f'{DRIVE_ROOT}/grazped_healing.pt'

IMG_224 = 224
IMG_384 = 384

LOCATIONS = ['wrist', 'forearm', 'humerus', 'tibia', 'ankle', 'clavicle', 'femur']

SEVERITY_DESC = {
    0: 'Grade 0 — Хагарал байхгүй',
    1: 'Grade 1 — Нарийн / hairline хагарал',
    2: 'Grade 2 — Энгийн бүрэн хагарал',
    3: 'Grade 3 — Хүнд / олон хэсэгт хагарал',
}

RECOVERY_TABLE = {
    0: (0,  0,  'Хагарал байхгүй'),
    1: (2,  4,  'Hairline — 2-4 долоо хоног'),
    2: (6,  8,  'Энгийн хагарал — 6-8 долоо хоног'),
    3: (10, 16, 'Хүнд хагарал — 10-16 долоо хоног'),
}

# Checkpoint байгаа эсэх шалгах
for name, path in [('MURA', MURA_CKPT), ('Hybrid U-Net', HYBRID_CKPT), ('Healing', HEAL_CKPT)]:
    status = '✓' if os.path.exists(path) else '✗ ОЛДСОНГҮЙ'
    size   = f'{os.path.getsize(path)/1e6:.0f}MB' if os.path.exists(path) else ''
    print(f'  {status} {name:15s}: {path}  {size}')

In [ ]:
# ── Cell 6: Transform ────────────────────────────────────────────────────────
TF_224 = A.Compose([
    A.Resize(IMG_224, IMG_224),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])
TF_384 = A.Compose([
    A.Resize(IMG_384, IMG_384),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
IMAGENET_STD  = np.array([0.229, 0.224, 0.225])

def load_tensor(path, size):
    tf  = TF_224 if size == IMG_224 else TF_384
    img = np.array(Image.open(path).convert('RGB'))
    return tf(image=img)['image'].unsqueeze(0).to(DEVICE)

def tensor_to_rgb(x):
    """Tensor → numpy RGB [0,1] (visualisation)"""
    img = x[0].permute(1, 2, 0).cpu().numpy()
    return np.clip(img * IMAGENET_STD + IMAGENET_MEAN, 0, 1)

print('✓ Transform бэлэн')

In [ ]:
# ── Cell 7: Загварын тодорхойлолт ────────────────────────────────────────────

# 1a. MURA Classifier v1 — EfficientNet-B3, 2-class softmax (mura_classifier.pt)
class MURAClassifier(nn.Module):
    def __init__(self, n_classes=2, drop=0.3):
        super().__init__()
        self.encoder = timm.create_model(
            'efficientnet_b3', pretrained=False,
            features_only=True, out_indices=[4])
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(384, 256), nn.GELU(), nn.Dropout(drop),
            nn.Linear(256, n_classes),
        )
    def forward(self, x):
        return self.head(self.encoder(x)[0])


# 1b. Fracture Classifier v2 — ConvNeXt/EfficientNetV2, binary sigmoid
class FractureClassifierV2(nn.Module):
    def __init__(self, encoder_name='convnext_small.in22k', drop=0.3):
        super().__init__()
        self.encoder = timm.create_model(
            encoder_name, pretrained=False, num_classes=0, drop_rate=drop)
        feat_dim = self.encoder.num_features
        self.head = nn.Sequential(
            nn.Linear(feat_dim, 512), nn.LayerNorm(512), nn.GELU(), nn.Dropout(drop),
            nn.Linear(512, 256), nn.GELU(), nn.Dropout(drop * 0.5),
            nn.Linear(256, 1),
        )
    def forward(self, x):
        return self.head(self.encoder(x))


# 2. Healing Model
class HealingHead(nn.Module):
    def __init__(self, in_dim=384, meta_dim=4, drop=0.2):
        super().__init__()
        self.meta_emb = nn.Sequential(nn.Linear(meta_dim, 16), nn.ReLU(True))
        cdim = in_dim + 16
        self.reg = nn.Sequential(
            nn.Linear(cdim, 256), nn.LayerNorm(256), nn.ReLU(True), nn.Dropout(drop),
            nn.Linear(256, 128), nn.LayerNorm(128), nn.ReLU(True), nn.Dropout(drop),
            nn.Linear(128, 64),  nn.ReLU(True), nn.Linear(64, 1),
        )
        self.log_var = nn.Sequential(
            nn.Linear(cdim, 64), nn.ReLU(True), nn.Linear(64, 1))
    def forward(self, feat, meta):
        g = F.adaptive_avg_pool2d(feat, 1).flatten(1)
        c = torch.cat([g, self.meta_emb(meta)], dim=1)
        return self.reg(c), self.log_var(c)

class HealingModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder   = timm.create_model(
            'efficientnet_b3', pretrained=False,
            features_only=True, out_indices=[4])
        self.heal_head = HealingHead(in_dim=384, meta_dim=4)
    def forward(self, x, meta):
        return self.heal_head(self.encoder(x)[0], meta)

print('✓ Загварын класс тодорхойлогдлоо (v1 + v2 дэмжинэ)')

In [ ]:
# ── Cell 8: Загваруудыг ачаалах ──────────────────────────────────────────────

def load_mura(path):
    """Checkpoint config-аас автоматаар v1 (EffB3) / v2 (ConvNeXt) таних."""
    ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
    cfg_saved = ckpt.get('config', {})
    encoder   = cfg_saved.get('encoder', '')
    img_size  = int(cfg_saved.get('img_size', IMG_224))

    if encoder and encoder != 'efficientnet_b3':
        m = FractureClassifierV2(encoder_name=encoder).to(DEVICE)
        m.load_state_dict(ckpt['model'])
        m.eval()
        print(f'  [OK] Classifier v2 ({encoder})  img={img_size}')
        return m, img_size, 'v2'
    else:
        m = MURAClassifier().to(DEVICE)
        m.load_state_dict(ckpt['model'])
        m.eval()
        print('  [OK] MURA Classifier v1 (EfficientNet-B3)  img=224')
        return m, IMG_224, 'v1'

def load_hybrid(path):
    m = smp.Unet(
        encoder_name='resnet50', encoder_weights=None,
        in_channels=3, classes=1).to(DEVICE)
    state = torch.load(path, map_location=DEVICE, weights_only=False)
    if isinstance(state, dict) and 'model' in state:
        state = state['model']
    m.load_state_dict(state)
    m.eval()
    return m

def load_healing(path):
    m = HealingModel().to(DEVICE)
    ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
    m.load_state_dict(ckpt['model'])
    m.eval()
    return m

print('Загваруудыг ачааллаж байна...')
mura_model, CLS_IMG_SIZE, CLS_VER = load_mura(MURA_CKPT)
hybrid_model = load_hybrid(HYBRID_CKPT)
print('  [OK] Hybrid ResNet U-Net  (Dice=0.604, TTA)')
heal_model   = load_healing(HEAL_CKPT)
print('  [OK] Healing Regressor    (EfficientNet-B3 + meta)')
print(f'\n✓ Бүх загвар бэлэн!  [{CLS_VER}  img={CLS_IMG_SIZE}]')

In [ ]:
# ── Cell 9: Inference функцүүд ────────────────────────────────────────────────

@torch.inference_mode()
def step1_classify(x, version='v1', threshold=0.5):
    logits = mura_model(x)
    if version == 'v2':
        frac_prob = float(torch.sigmoid(logits.squeeze()).item())
        norm_prob  = 1.0 - frac_prob
        label      = int(frac_prob >= threshold)
    else:
        probs_list = F.softmax(logits, dim=1)[0].cpu().tolist()
        norm_prob, frac_prob = probs_list[0], probs_list[1]
        label = int(logits.argmax(1).item())
    return {
        'fracture':      bool(label == 1),
        'confidence':    round(float(max(frac_prob, norm_prob)), 4),
        'prob_normal':   round(norm_prob, 4),
        'prob_fracture': round(frac_prob, 4),
    }


@torch.inference_mode()
def step2_segment(x):
    """4-fold TTA: orig + hflip + vflip + hvflip"""
    probs = []
    for flip in [None, [-1], [-2], [-1, -2]]:
        xi = x if flip is None else torch.flip(x, dims=flip)
        pi = torch.sigmoid(hybrid_model(xi))
        if flip is not None:
            pi = torch.flip(pi, dims=flip)
        probs.append(pi)
    prob_t  = torch.stack(probs).mean(0)
    prob_np = prob_t[0, 0].cpu().numpy()
    mask    = (prob_np > 0.5).astype(np.uint8)
    return mask, prob_np


def step3_severity(mask):
    h, w = mask.shape
    feats = dict(area_px=0., area_pct=0., n_components=0,
                 largest_comp_pct=0., major_axis=0., minor_axis=0.,
                 aspect_ratio=0., eccentricity=0., solidity=0., bone_region=1)
    if mask.sum() == 0:
        grade = 0
    else:
        feats['area_px']  = float(mask.sum())
        feats['area_pct'] = float(mask.sum() / (h * w) * 100)
        labeled, n = ndimage.label(mask)
        feats['n_components'] = int(n)
        if n > 0:
            sizes = ndimage.sum(mask, labeled, range(1, n + 1))
            feats['largest_comp_pct'] = float(np.max(sizes) / mask.sum())
        props = measure.regionprops(labeled)
        if props:
            biggest = max(props, key=lambda p: p.area)
            feats['major_axis']   = float(biggest.major_axis_length)
            feats['minor_axis']   = float(biggest.minor_axis_length)
            feats['eccentricity'] = float(biggest.eccentricity)
            feats['solidity']     = float(biggest.solidity)
            if biggest.minor_axis_length > 1e-6:
                feats['aspect_ratio'] = float(
                    biggest.major_axis_length / biggest.minor_axis_length)
            cy, _ = biggest.centroid
            feats['bone_region'] = 0 if cy/h < 0.33 else (2 if cy/h > 0.67 else 1)
        a, n_c, sol = feats['area_pct'], feats['n_components'], feats['solidity']
        if feats['area_px'] < 30:                           grade = 0
        elif a > 5. or n_c >= 3 or (0 < sol < 0.65):       grade = 3
        elif a > 1. or n_c == 2:                            grade = 2
        else:                                               grade = 1

    rec_lo, rec_hi, rec_desc = RECOVERY_TABLE[grade]
    return {
        'grade': grade, 'label': SEVERITY_DESC[grade],
        'area_pct': round(feats['area_pct'], 3),
        'n_pieces': feats['n_components'],
        'solidity': round(feats['solidity'], 3),
        'recovery_min': rec_lo, 'recovery_max': rec_hi,
        'recovery_desc': rec_desc,
    }


@torch.inference_mode()
def step4_healing(x, age, sex, location, severity):
    loc_id = (float(LOCATIONS.index(location)) / len(LOCATIONS)
              if location in LOCATIONS else 0.)
    meta = torch.tensor(
        [[age / 100., 1. if str(sex).upper() == 'M' else 0.,
          loc_id, float(severity) / 3.]],
        dtype=torch.float32, device=DEVICE)
    pred, log_var = heal_model(x, meta)
    weeks = float(pred.item())
    std   = float(torch.exp(0.5 * log_var).item())
    return {
        'weeks': round(weeks, 1),
        'ci_95': [round(max(0., weeks - 1.96 * std), 1),
                  round(weeks + 1.96 * std, 1)],
        'std':   round(std, 2),
    }

print('✓ Inference функцүүд бэлэн')

In [ ]:
# ── Cell 10: Бүрэн pipeline функц ────────────────────────────────────────────

def run_pipeline(image_path, age=25, sex='M', location='wrist', force=False):
    """
    Нэг рентген зургаар бүрэн pipeline ажиллуулна.

    Args:
        image_path : str — зургийн зам
        age        : int — нас
        sex        : str — 'M' эсвэл 'F'
        location   : str — wrist / forearm / humerus / tibia / ankle / clavicle / femur
        force      : bool — ангилал 'хагарал байхгүй' гарсан ч үргэлжлүүл

    Note:
        CLS_IMG_SIZE, CLS_VER — Cell 8-д ачаалагдах үед автоматаар тодорхойлогдно.
        v1 (mura_classifier.pt)        → IMG_SIZE=224, softmax
        v2 (fracture_classifier_v2.pt) → IMG_SIZE=384, sigmoid
    """
    x_cls = load_tensor(image_path, CLS_IMG_SIZE)  # v1→224, v2→384
    x384  = load_tensor(image_path, IMG_384)

    result = {
        'image':   image_path,
        'patient': {'age': age, 'sex': sex, 'location': location},
    }

    # Алхам 1: Ангилал
    cls = step1_classify(x_cls, version=CLS_VER)
    result['classification'] = cls

    if not cls['fracture'] and not force:
        result['segmentation'] = None
        result['severity']     = None
        result['healing']      = None
        return result, x384, None, None

    # Алхам 2: Сегментчлэл (Hybrid U-Net + TTA)
    mask, prob_np = step2_segment(x384)
    result['segmentation'] = {'mask_coverage_pct': round(float(mask.mean() * 100), 2)}

    # Алхам 3: Severity (morphological heuristic)
    sev = step3_severity(mask)
    result['severity'] = sev

    # Алхам 4: Эдгэрэлтийн таамаглал
    x_heal = load_tensor(image_path, IMG_224)
    heal   = step4_healing(x_heal, age, sex, location, sev['grade'])
    result['healing'] = heal

    return result, x384, mask, prob_np

print('✓ Pipeline функц бэлэн')

In [ ]:
# ── Cell 11: Визуализацийн функц ─────────────────────────────────────────────

GRADE_COLORS = {0: '#2ecc71', 1: '#f1c40f', 2: '#e67e22', 3: '#e74c3c'}

def visualize(result, x384, mask, prob_np):
    cls  = result['classification']
    sev  = result.get('severity')
    heal = result.get('healing')
    img_rgb = tensor_to_rgb(x384)  # (384,384,3)

    if mask is None:
        fig, ax = plt.subplots(1, 1, figsize=(5, 5))
        ax.imshow(img_rgb)
        ax.set_title(f'Хагарал байхгүй  ({cls["confidence"]:.1%})',
                     color='green', fontsize=13, fontweight='bold')
        ax.axis('off')
        plt.tight_layout()
        plt.show()
        return

    grade = sev['grade']
    color = GRADE_COLORS[grade]

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    fig.patch.set_facecolor('#1a1a2e')

    # 1) Эх зураг
    axes[0].imshow(img_rgb)
    axes[0].set_title('Рентген зураг', color='white', fontsize=11)

    # 2) Маск overlay
    overlay = img_rgb.copy()
    overlay[mask == 1] = overlay[mask == 1] * 0.4 + np.array([1., 0.2, 0.1]) * 0.6
    axes[1].imshow(overlay)
    status = 'ХАГАРАЛ' if cls['fracture'] else f'force ({cls["prob_fracture"]:.0%})'
    axes[1].set_title(
        f'Сегментчлэл — {sev["area_pct"]:.2f}%\n{status}',
        color='#e74c3c' if cls['fracture'] else '#f39c12', fontsize=10)

    # 3) Probability heatmap
    axes[2].imshow(img_rgb)
    axes[2].imshow(prob_np, cmap='hot', alpha=0.6, vmin=0, vmax=1)
    axes[2].set_title('Probability heatmap', color='white', fontsize=11)

    # 4) Үр дүн хуудас
    axes[3].set_facecolor('#0d1117')
    axes[3].axis('off')
    lo, hi = heal['ci_95']
    lines = [
        (f'{sev["label"]}',                  color,      14, 'bold'),
        ('',                                  'white',     8, 'normal'),
        (f'Маск хамрах: {sev["area_pct"]:.2f}%', '#aaa', 10, 'normal'),
        (f'Хэсгийн тоо: {sev["n_pieces"]}',  '#aaa',    10, 'normal'),
        (f'Solidity:    {sev["solidity"]:.3f}','#aaa',   10, 'normal'),
        ('',                                  'white',     8, 'normal'),
        (f'Эдгэрэлт: {heal["weeks"]} долоо хоног', '#3498db', 12, 'bold'),
        (f'95% CI: {lo} – {hi} дх',          '#7fb3d3',  10, 'normal'),
        (f'σ = {heal["std"]} дх',             '#7fb3d3',  10, 'normal'),
        ('',                                  'white',     8, 'normal'),
        (sev['recovery_desc'],                color,      10, 'italic'),
    ]
    y = 0.95
    for txt, clr, fs, fw in lines:
        axes[3].text(0.05, y, txt, transform=axes[3].transAxes,
                     color=clr, fontsize=fs, fontweight=fw, va='top')
        y -= 0.09

    for ax in axes:
        ax.set_facecolor('#1a1a2e')
        ax.tick_params(left=False, labelleft=False,
                       bottom=False, labelbottom=False)
        for spine in ax.spines.values():
            spine.set_edgecolor('#333')

    fname = Path(result['image']).name
    pt    = result['patient']
    fig.suptitle(
        f'{fname}  |  нас={pt["age"]}  хүйс={pt["sex"]}  байршил={pt["location"]}',
        color='white', fontsize=12, y=1.01)
    plt.tight_layout()
    plt.show()

print('✓ Визуализацийн функц бэлэн')

---
## 🔬 Нэг зурагт туршиx

Доорх `IMAGE_PATH`, `AGE`, `SEX`, `LOCATION`-ыг өөрчил.

In [ ]:
# ── Cell 12: Нэг зураг ───────────────────────────────────────────────────────
#
# Зургийн замыг энд тохируул:
IMAGE_PATH = '/content/drive/MyDrive/test_xray.jpg'  # ← ӨӨРИЙН ЗУРАГНЫ ЗАМ
AGE        = 25
SEX        = 'M'    # 'M' эсвэл 'F'
LOCATION   = 'wrist'  # wrist / forearm / humerus / tibia / ankle / clavicle / femur
FORCE      = True   # True → ангилал 'хагарал байхгүй' гарсан ч бусад алхмуудыг ажиллуул

result, x384, mask, prob_np = run_pipeline(
    IMAGE_PATH, age=AGE, sex=SEX, location=LOCATION, force=FORCE)

visualize(result, x384, mask, prob_np)

# Тоон үр дүн хэвлэх
cls  = result['classification']
sev  = result.get('severity') or {}
heal = result.get('healing') or {}

print('=' * 55)
print(f'  [1] Ангилал  : {"ХАГАРАЛ" if cls["fracture"] else "Хагарал байхгүй"}')
print(f'       Итгэл   : {cls["confidence"]:.1%}')
print(f'       Магадлал: хэвийн={cls["prob_normal"]:.3f}  хагарал={cls["prob_fracture"]:.3f}')
if sev:
    print(f'  [2] Сегмент  : маск {result["segmentation"]["mask_coverage_pct"]:.2f}%')
    print(f'  [3] Severity : {sev["label"]}')
    print(f'       Хэсэг   : {sev["n_pieces"]}  |  Solidity: {sev["solidity"]:.3f}')
    print(f'       Эдгэрэлт: {sev["recovery_min"]}-{sev["recovery_max"]} долоо хоног')
if heal:
    print(f'  [4] Загвар   : {heal["weeks"]} дх  (95% CI: {heal["ci_95"][0]}–{heal["ci_95"][1]},  σ={heal["std"]})')
print('=' * 55)

---
## 📁 Folder дахь бүх зургийг боловсруулах

In [ ]:
# ── Cell 13: Batch inference ─────────────────────────────────────────────────
FOLDER     = '/content/drive/MyDrive/test_images'  # ← зургийн folder
BATCH_AGE  = 25
BATCH_SEX  = 'M'
BATCH_LOC  = 'wrist'
BATCH_FORCE = True
OUTPUT_JSON = '/content/drive/MyDrive/pipeline_results.json'

exts = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}
paths = [str(p) for p in Path(FOLDER).iterdir() if p.suffix.lower() in exts]
print(f'{len(paths)} зураг олдлоо')

all_results = []
for i, path in enumerate(paths, 1):
    print(f'  [{i}/{len(paths)}] {Path(path).name}', end=' ... ')
    try:
        r, x384, mask, prob_np = run_pipeline(
            path, age=BATCH_AGE, sex=BATCH_SEX,
            location=BATCH_LOC, force=BATCH_FORCE)
        sev  = r.get('severity') or {}
        heal = r.get('healing') or {}
        print(f'Grade {sev.get("grade", "-")}  |  {heal.get("weeks", "-")} дх')
        visualize(r, x384, mask, prob_np)
    except Exception as e:
        print(f'АЛДАА: {e}')
        r = {'image': path, 'error': str(e)}
    all_results.append(r)

# JSON хадгалах
with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)
print(f'\n✓ Үр дүн хадгалагдлаа → {OUTPUT_JSON}')

---
## 📊 Batch үр дүнгийн нэгтгэл

In [ ]:
# ── Cell 14: Batch нэгтгэл ───────────────────────────────────────────────────
import pandas as pd
from collections import Counter

rows = []
for r in all_results:
    if 'error' in r:
        continue
    cls  = r.get('classification', {})
    sev  = r.get('severity') or {}
    heal = r.get('healing') or {}
    rows.append({
        'filename':        Path(r['image']).name,
        'fracture':        cls.get('fracture', False),
        'cls_confidence':  cls.get('confidence', 0),
        'severity_grade':  sev.get('grade', None),
        'severity_label':  sev.get('label', ''),
        'mask_pct':        sev.get('area_pct', 0),
        'n_pieces':        sev.get('n_pieces', 0),
        'healing_weeks':   heal.get('weeks', None),
        'ci_lo':           heal.get('ci_95', [None])[0],
        'ci_hi':           heal.get('ci_95', [None, None])[1],
    })

df = pd.DataFrame(rows)
print(f'Нийт: {len(df)} зураг\n')
print(df[['filename', 'severity_grade', 'mask_pct', 'healing_weeks']].to_string(index=False))

# Severity distribution
if not df.empty and 'severity_grade' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    grade_counts = df['severity_grade'].value_counts().sort_index()
    colors = [GRADE_COLORS.get(g, 'gray') for g in grade_counts.index]
    axes[0].bar(grade_counts.index.astype(str), grade_counts.values, color=colors)
    axes[0].set_title('Severity Grade тархалт')
    axes[0].set_xlabel('Grade')
    axes[0].set_ylabel('Зургийн тоо')

    heal_valid = df['healing_weeks'].dropna()
    if not heal_valid.empty:
        axes[1].hist(heal_valid, bins=10, color='#3498db', edgecolor='white')
        axes[1].set_title('Эдгэрэлтийн хугацааны тархалт')
        axes[1].set_xlabel('Долоо хоног')
        axes[1].set_ylabel('Зургийн тоо')

    plt.tight_layout()
    plt.show()